# Python — ETL Patterns: argparse, Logging, Pipeline Structure
---

<div style="padding:15px;border-left:8px solid #fa709a;background:#fff0f5;border-radius:4px;">
<strong>Core Insight:</strong> Production ETL pipelines are not scripts — they are programs.
They accept arguments, emit structured logs, handle errors gracefully, and exit with
the right code for orchestrators (Airflow, Kubernetes). Learn this structure once,
apply it to every pipeline you write.
</div>

### The Citi Pattern
The ETL pipeline template came directly from Citi's capacity data pipeline —
`--env`, `--date`, `--dry-run` flags, structured JSON logging, `sys.exit(1)` on failure
for Airflow retry detection. The same code ran in dev, staging, and production with zero
code changes — only environment variable differed.

## 🧠 The Production ETL Checklist

| Concern | The Pattern |
|---------|------------|
| **Arguments** | `argparse` — not hardcoded values. `--env prod`, `--date 2024-01-15`, `--dry-run` |
| **Logging** | Structured JSON logs — not `print()`. Level + timestamp + context in every message |
| **Error Handling** | `try/except` at pipeline level. `sys.exit(1)` on failure → Airflow retries |
| **Idempotency** | Running the pipeline twice for the same `--date` produces the same result |
| **Dry Run** | `--dry-run` flag to validate without writing — essential for testing in prod |
| **Configuration** | Environment variables for secrets. `argparse` for run parameters |

### Why Structured Logs?
`print("Processing complete")` is useless in production.
`{"ts": "2024-01-15T14:23:00Z", "level": "INFO", "job": "capacity-etl", "rows": 14400, "env": "prod"}`
can be queried in CloudWatch Logs, Splunk, or ELK — automatically parsed, alertable, searchable.

In [ ]:
#!/usr/bin/env python3
"""
capacity_etl.py — Production-grade ETL pipeline template
Usage: python capacity_etl.py --env prod --date 2024-01-15 [--dry-run]
"""
import argparse
import json
import logging
import sys
import time
from datetime import date, datetime


# ── Structured logging setup ─────────────────────────────────────────────────

class JSONFormatter(logging.Formatter):
    """Emit logs as JSON lines — compatible with CloudWatch, ELK, Splunk."""
    def __init__(self, job_name: str):
        super().__init__()
        self.job_name = job_name

    def format(self, record: logging.LogRecord) -> str:
        log_data = {
            "ts": datetime.utcnow().isoformat() + "Z",
            "level": record.levelname,
            "job": self.job_name,
            "msg": record.getMessage(),
        }
        # Add any extra fields passed to the logger
        for key, value in record.__dict__.items():
            if key not in ("msg", "args", "levelname", "levelno", "pathname",
                           "filename", "module", "exc_info", "exc_text",
                           "stack_info", "lineno", "funcName", "created",
                           "msecs", "relativeCreated", "thread", "threadName",
                           "processName", "process", "name", "message"):
                log_data[key] = value
        return json.dumps(log_data)


def setup_logging(job_name: str, level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger(job_name)
    logger.setLevel(getattr(logging, level))
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(JSONFormatter(job_name))
    logger.addHandler(handler)
    logger.propagate = False
    return logger

In [ ]:
# ── Argument parsing ─────────────────────────────────────────────────────────

def parse_args(argv=None):
    parser = argparse.ArgumentParser(
        description="Capacity ETL — extract monitoring data, load to data warehouse",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  %(prog)s --env prod --date 2024-01-15
  %(prog)s --env dev --date 2024-01-15 --dry-run
  %(prog)s --env staging --date 2024-01-15 --log-level DEBUG
        """
    )
    parser.add_argument(
        "--env", required=True, choices=["dev", "staging", "prod"],
        help="Target environment"
    )
    parser.add_argument(
        "--date", required=True, type=lambda s: date.fromisoformat(s),
        help="Date to process (YYYY-MM-DD)"
    )
    parser.add_argument(
        "--dry-run", action="store_true",
        help="Validate and log without writing any data"
    )
    parser.add_argument(
        "--log-level", default="INFO", choices=["DEBUG", "INFO", "WARNING", "ERROR"],
        help="Logging level (default: INFO)"
    )
    return parser.parse_args(argv)


# ── Pipeline stages ──────────────────────────────────────────────────────────

def extract(env: str, run_date: date, logger: logging.Logger) -> list:
    logger.info("Starting extract", extra={"stage": "extract", "env": env, "date": str(run_date)})
    # In production: connect to APM API or S3
    rows = [{"server_id": f"SRV-{i:04d}", "cpu": 45.0 + i*0.01, "date": str(run_date)}
            for i in range(1, 6001)]  # 6000 servers
    logger.info("Extract complete", extra={"stage": "extract", "rows": len(rows)})
    return rows


def transform(rows: list, logger: logging.Logger) -> list:
    logger.info("Starting transform", extra={"stage": "transform", "input_rows": len(rows)})
    transformed = []
    errors = 0
    for row in rows:
        try:
            transformed.append({
                "server_id": row["server_id"],
                "cpu_pct": round(float(row["cpu"]), 2),
                "date": row["date"],
                "alert": row["cpu"] > 80
            })
        except (KeyError, ValueError) as e:
            errors += 1
            logger.warning("Row transform failed", extra={"error": str(e), "row": str(row)[:100]})
    logger.info("Transform complete", extra={
        "stage": "transform", "output_rows": len(transformed), "errors": errors
    })
    return transformed


def load(rows: list, env: str, dry_run: bool, logger: logging.Logger) -> int:
    if dry_run:
        logger.info("DRY RUN: skipping load", extra={"stage": "load", "would_write": len(rows)})
        return len(rows)
    logger.info("Starting load", extra={"stage": "load", "env": env, "rows": len(rows)})
    # In production: write to S3/Redshift/Snowflake
    time.sleep(0.1)  # simulate I/O
    logger.info("Load complete", extra={"stage": "load", "rows_written": len(rows)})
    return len(rows)


# ── Main entrypoint ──────────────────────────────────────────────────────────

def main(argv=None):
    args = parse_args(argv)
    logger = setup_logging("capacity-etl", level=args.log_level)

    logger.info("Pipeline started", extra={
        "env": args.env, "date": str(args.date), "dry_run": args.dry_run
    })
    start = time.perf_counter()

    try:
        rows = extract(args.env, args.date, logger)
        rows = transform(rows, logger)
        written = load(rows, args.env, args.dry_run, logger)
        elapsed = round(time.perf_counter() - start, 2)
        logger.info("Pipeline complete", extra={
            "rows_written": written, "duration_s": elapsed, "env": args.env
        })
        return 0  # success — Airflow reads this exit code

    except Exception as e:
        elapsed = round(time.perf_counter() - start, 2)
        logger.error("Pipeline failed", extra={
            "error": str(e), "error_type": type(e).__name__, "duration_s": elapsed
        })
        return 1  # failure — Airflow will retry


if __name__ == "__main__":
    sys.exit(main())

# Demo run (as if called from command line):
exit_code = main(["--env", "dev", "--date", "2024-01-15", "--dry-run"])
print(f"\nExit code: {exit_code} (0=success, 1=failure)")

## 📊 Idempotency — The Critical Property

**Definition:** Running the pipeline multiple times with the same inputs produces the same result.

**Why it matters:** Airflow retries failed tasks. Without idempotency:
- Retry inserts duplicate rows → wrong aggregates
- Retry re-processes data that was already loaded → double-counting

**How to implement:**
```sql
-- Option 1: DELETE + INSERT (for date partitions)
DELETE FROM fact_monitoring WHERE date = :run_date;
INSERT INTO fact_monitoring SELECT ... WHERE date = :run_date;

-- Option 2: UPSERT (INSERT ... ON CONFLICT)
INSERT INTO fact_monitoring (server_id, date, cpu_pct)
VALUES (%(server_id)s, %(date)s, %(cpu_pct)s)
ON CONFLICT (server_id, date)
DO UPDATE SET cpu_pct = EXCLUDED.cpu_pct;

-- Option 3: Overwrite S3 partition
s3://bucket/monitoring/date=2024-01-15/  # overwrite entire partition
```

**The golden rule:** A pipeline that can be safely retried is a pipeline that can be operated.
Every production pipeline must be idempotent.

## 🎤 5 Interview Q&A

**Q1: Why use `sys.exit(1)` instead of raising an exception?**
Orchestrators like Airflow, Kubernetes, and shell scripts check the process exit code — not Python exceptions.
`sys.exit(1)` signals failure to the OS. `sys.exit(0)` signals success.
If you raise an unhandled exception, the process exits with code 1 anyway, but the error message
may not be in your structured logs. Catching the exception, logging it with full context,
then calling `sys.exit(1)` gives you both the correct exit code and a searchable log entry.

---

**Q2: What is idempotency and how do you achieve it in an ETL pipeline?**
An idempotent pipeline produces the same output when run multiple times with the same input.
Achieve it by partitioning your target table by the processing date and using DELETE+INSERT
or UPSERT on each run. Airflow retries mean pipelines run multiple times — idempotency
ensures duplicates don't accumulate.

---

**Q3: Why structured JSON logs instead of print statements?**
JSON logs are machine-parseable. CloudWatch Logs Insights, Splunk, and ELK can parse them
automatically, enabling queries like "show all runs where rows > 10000 and env = prod".
Print statements are strings — you can't filter them programmatically without regex.
In a production system with 50 pipelines running daily, structured logs are how you debug
at 3am without SSH access to the server.

---

**Q4: What is a dry-run flag and when do you use it?**
`--dry-run` executes all pipeline logic except the final write operation. It extracts,
transforms, validates — but skips the load step. Use cases: (1) test a new pipeline
against production data without mutating anything; (2) validate a backfill before running
it; (3) debug issues by seeing what WOULD be written. Critical in production environments
where "undo" is not an option.

---

**Q5: What is the difference between `logging.info()` and `print()`?**
`print()` is ephemeral — it goes to stdout with no metadata.
`logging.info()` goes through the logging framework with: timestamp, level, logger name, module,
line number, and any `extra={}` fields you add. The logging framework supports:
handlers (route to file, stdout, CloudWatch), formatters (plain text vs JSON), and levels
(filter debug noise in production with `logging.WARNING`). In production, always use logging.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **ETL** | Extract, Transform, Load — the pattern for moving data from source to destination |
| **ELT** | Extract, Load, Transform — load raw data first, transform in-warehouse (modern data platform approach) |
| **Idempotency** | Running the same operation multiple times produces the same result — critical for retry safety |
| **argparse** | Python standard library for CLI argument parsing — `--flag value`, `--boolean-flag` |
| **Structured Logging** | Logs as JSON/key-value pairs — machine-parseable, filterable in log aggregation tools |
| **Exit Code** | Integer returned by a process: 0=success, non-zero=failure — orchestrators use this for retry logic |
| **Dry Run** | Execute logic without side effects — validates without writing data |
| **Backfill** | Re-running a pipeline for past dates to populate historical data |
| **Orchestrator** | Tool that schedules and monitors pipelines (Airflow, Prefect, Dagster, Kubernetes CronJobs) |
| **Partition** | Organizing data by a key (usually date) so loads can be targeted and overwritten safely |

## 💼 The Citi Narrative

**Context:** Citi capacity ETL pipeline — extracts APM data for 6,000 servers,
transforms into daily capacity summaries, loads to PostgreSQL for reporting.

**The Problem:** Original script had hardcoded database credentials, printed status to stdout,
and would silently succeed even if 0 rows were written. Airflow marked it as successful
when it shouldn't have. Debugging failures required SSH access and grep-ing through logs.

**The Fix — Four Changes:**
1. `argparse`: `--env prod/dev`, `--date 2024-01-15`, `--dry-run` flags
2. JSON logging: every stage logged rows in/out, duration, any errors
3. `sys.exit(1)` on pipeline failure: Airflow retries automatically
4. Idempotent load: `DELETE WHERE date = :run_date` + `INSERT` — safe to retry

**Result:** Same pipeline code ran in dev, staging, and production with zero code changes.
Airflow retry worked correctly. When a source DB was unavailable at 06:00, the pipeline
retried at 06:05, succeeded, and no manual intervention was needed.

**Interview Line:** *"The difference between a script and a pipeline is one of these four things.
A script prints to stdout and exits. A pipeline logs JSON, accepts `--env` and `--date`,
handles errors with the right exit code, and can be retried safely."*

In [ ]:
# Practice: Fix the anti-pattern pipeline below

import sys, time

# ❌ ANTI-PATTERN: Script, not a pipeline
def bad_pipeline():
    env = "prod"                              # hardcoded
    date = "2024-01-15"                       # hardcoded
    print("Starting ETL for", date)           # not structured

    try:
        data = [{"id": i, "val": i*1.5} for i in range(100)]
        print("Got", len(data), "rows")       # not structured
        # load to DB...
        print("Done")                         # no exit code
    except Exception as e:
        print("Error:", e)                    # not structured, no exit

# ✅ YOUR TASK: Rewrite using the patterns from this notebook
# Requirements:
# 1. Accept --env and --date via argparse
# 2. Log each step as JSON with level, msg, stage, rows
# 3. Return exit code 0 on success, 1 on failure
# 4. Support --dry-run flag

# See the full solution in cells c03-c04 above.
# Key test: run with --dry-run and verify no DB writes occur.
# Run with invalid --env and verify argparse rejects it.

print("Anti-pattern identified. See cells c03-c04 for the correct pattern.")
print("Key insight: same 4 changes apply to ANY ETL pipeline you write.")

## 🎯 Summary

### The Production ETL Template
```
main()
  ├── parse_args()        → argparse: --env, --date, --dry-run
  ├── setup_logging()     → JSON formatter, stdout handler
  ├── try:
  │   ├── extract()       → log rows extracted
  │   ├── transform()     → log rows in/out, errors
  │   └── load()          → skip if --dry-run
  └── except:
      └── log error → sys.exit(1)   ← Airflow retries
```

### Interview Confidence Checklist
- [ ] Can explain why sys.exit(1) matters for orchestrators
- [ ] Can explain idempotency and give a SQL implementation
- [ ] Can write a JSON formatter for Python logging
- [ ] Can describe argparse with --env, --date, --dry-run
- [ ] Can name the Citi narrative: same code in dev/staging/prod with zero changes

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀